In [ ]:
%pip install ipywidgets pillow

In [ ]:
import io, warnings
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from scipy.ndimage import rotate as nd_rotate, gaussian_filter, map_coordinates
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, Layout, Output
from IPython.display import display, Image, clear_output
from PIL import Image as PILImage
warnings.filterwarnings('ignore')

# Shared layout and label-width settings for all interactive sliders
slider_layout = Layout(width="420px")
label_style   = {"description_width": "160px"}


def figure_to_image(fig, dpi=100):
    """Render a matplotlib figure to a PNG and return it as an IPython Image for notebook display."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    return Image(data=buf.read())


def make_phantom_image(size=100):
    """Create a 2-D phantom image that mimics a biological cross-section.

    The phantom contains four features:
      - A hollow ring (simulating a microtubule cross-section)
      - Two perpendicular line segments (simulating membrane sheets)
      - A small filled disc (simulating a protein complex)

    All features have intensities between 0 and 1. The image is returned
    with the y-axis flipped so the origin is at the bottom-left (standard
    mathematical convention).
    """
    y, x = np.mgrid[0:size, 0:size].astype(float)
    image = np.zeros((size, size))

    # Feature 1: hollow ring in the upper-left quadrant
    ring_cx, ring_cy = size * 0.33, size * 0.33
    ring_radius = np.sqrt((x - ring_cx) ** 2 + (y - ring_cy) ** 2)
    image += 0.9 * ((ring_radius > size * 0.12) & (ring_radius < size * 0.18))

    # Feature 2: horizontal line segment (membrane-like)
    image += 0.85 * (np.abs(y - size * 0.68) < size * 0.018)

    # Feature 3: vertical line segment (membrane-like)
    image += 0.85 * (np.abs(x - size * 0.67) < size * 0.018) * ((y > size * 0.4) & (y < size * 0.9))

    # Feature 4: small filled disc (dense protein complex)
    disc_cx, disc_cy = size * 0.72, size * 0.35
    disc_radius = np.sqrt((x - disc_cx) ** 2 + (y - disc_cy) ** 2)
    image += 0.75 * (disc_radius < size * 0.06)

    return np.clip(image, 0, 1)[::-1]   # flip y for standard orientation


# ---------------------------------------------------------------------------
# Tomographic projection and reconstruction algorithms
# ---------------------------------------------------------------------------

def compute_radon_projections(image, tilt_angles_deg):
    """Compute the Radon transform (tilt series) of an image.

    For each tilt angle α, the specimen is rotated by α and then the
    columns are summed to produce a 1-D projection. This simulates the
    physical process of tilting the specimen in the electron microscope.

    Parameters
    ----------
    image           : 2-D array representing the specimen cross-section
    tilt_angles_deg : 1-D array of tilt angles in degrees

    Returns
    -------
    sinogram : 2-D array of shape (len(tilt_angles_deg), image_width)
               Each row is the 1-D projection at the corresponding tilt angle.
    """
    sinogram = np.zeros((len(tilt_angles_deg), image.shape[1]))
    for row_index, tilt_angle in enumerate(tilt_angles_deg):
        rotated_specimen = nd_rotate(image, tilt_angle, reshape=False, cval=0)
        sinogram[row_index, :] = rotated_specimen.sum(axis=0)
    return sinogram


def simple_backprojection(sinogram, tilt_angles_deg):
    """Reconstruct an image from a sinogram using simple (unfiltered) backprojection.

    For each projection at angle α:
      1. Repeat the 1-D projection into a 2-D slab of the same height as the
         reconstruction grid.
      2. Rotate the slab back by −α.
      3. Add it to the running sum.
    Average over all angles.

    This is mathematically correct but overweights low spatial frequencies
    because every projection contributes equally to the DC component. The
    result is a blurry, star-shaped artefact around each feature.
    """
    grid_size = sinogram.shape[1]
    reconstruction = np.zeros((grid_size, grid_size))
    for projection, tilt_angle in zip(sinogram, tilt_angles_deg):
        # Tile the 1-D projection into a 2-D slab, then rotate back
        projection_slab = np.tile(projection[np.newaxis, :], (grid_size, 1)) / grid_size
        reconstruction += nd_rotate(projection_slab, -tilt_angle, reshape=False, cval=0)
    return reconstruction / max(len(tilt_angles_deg), 1)


def apply_ramp_filter(sinogram):
    """Pre-filter each projection in the sinogram with the ramp filter |ν|.

    The ramp filter compensates for the 1/|ν| low-frequency bias introduced
    by backprojection (see `simple_backprojection`). Multiplying the Fourier
    transform of each 1-D projection by |ν| before backprojecting yields an
    exact reconstruction (for complete angular coverage).
    """
    num_samples = sinogram.shape[1]
    ramp_freqs  = np.fft.rfftfreq(num_samples)   # frequency axis: 0 to 0.5
    filtered_sinogram = np.zeros_like(sinogram)
    for row_index, projection in enumerate(sinogram):
        fourier_projection = np.fft.rfft(projection)
        filtered_sinogram[row_index] = np.fft.irfft(fourier_projection * ramp_freqs, num_samples)
    return filtered_sinogram


FBP_FILTERS = {
    'ramp':        'Ram-Lak (ramp)',
    'shepp-logan': 'Shepp-Logan',
    'cosine':      'Cosine',
    'hamming':     'Hamming',
    'hann':        'Hann',
}


def apply_fbp_filter(sinogram, filter_name='ramp'):
    """Pre-filter each projection with a chosen FBP filter and return the
    filtered sinogram together with the filter's frequency response.

    All options are windowed versions of the ramp filter |ν| (see
    `apply_ramp_filter`): the window tapers the ramp towards zero near the
    Nyquist frequency, trading a little resolution for less amplification of
    high-frequency noise. From least to most tapered: 'ramp' (no window,
    sharpest but noisiest), 'shepp-logan', 'cosine', 'hamming', 'hann'
    (smoothest, most attenuated at high frequency).
    """
    num_samples = sinogram.shape[1]
    freqs   = np.fft.rfftfreq(num_samples)   # frequency axis: 0 to 0.5
    ramp    = freqs.copy()
    nyquist = freqs[-1] if freqs[-1] > 0 else 1.0

    if filter_name == 'ramp':
        window = np.ones_like(freqs)
    elif filter_name == 'shepp-logan':
        window = np.sinc(freqs / (2 * nyquist))
    elif filter_name == 'cosine':
        window = np.cos(np.pi * freqs / (2 * nyquist))
    elif filter_name == 'hamming':
        window = 0.54 + 0.46 * np.cos(np.pi * freqs / nyquist)
    elif filter_name == 'hann':
        window = 0.5 * (1 + np.cos(np.pi * freqs / nyquist))
    else:
        raise ValueError(f"Unknown filter '{filter_name}'; choose from {list(FBP_FILTERS)}")

    filter_response = ramp * window
    filtered_sinogram = np.zeros_like(sinogram)
    for row_index, projection in enumerate(sinogram):
        fourier_projection = np.fft.rfft(projection)
        filtered_sinogram[row_index] = np.fft.irfft(fourier_projection * filter_response, num_samples)
    return filtered_sinogram, filter_response, freqs


def filtered_backprojection(sinogram, tilt_angles_deg):
    """Reconstruct an image using filtered backprojection (FBP).

    Applies the ramp filter to each projection before backprojecting.
    FBP produces a sharper reconstruction than simple backprojection and
    is the analytical gold standard for complete (full-angle) tilt series.
    """
    return simple_backprojection(apply_ramp_filter(sinogram), tilt_angles_deg)


def power_spectrum_2d(image):
    """Compute the log-scaled 2-D power spectrum, with zero frequency at the centre."""
    fourier_transform = np.fft.fftshift(np.fft.fft2(image))
    return np.log(np.abs(fourier_transform) ** 2 + 1)


# ---------------------------------------------------------------------------
# Fourier slice theorem helper
# ---------------------------------------------------------------------------

def extract_central_fourier_slice(fourier_2d, tilt_angle_deg):
    """Extract a central slice from a 2-D Fourier transform at the given angle.

    The Fourier slice theorem states that the 1-D Fourier transform of a
    projection at angle α equals the central slice through the 2-D Fourier
    transform at the same angle. This function extracts that slice using
    bilinear interpolation.

    Parameters
    ----------
    fourier_2d      : 2-D complex array — the 2-D Fourier transform of the object
    tilt_angle_deg  : tilt angle in degrees

    Returns a 1-D complex array — the central slice values.
    """
    n = fourier_2d.shape[0]
    angle_rad = np.deg2rad(tilt_angle_deg)
    # Parametric line through the origin at the given angle
    t = np.linspace(-n // 2, n // 2, n)
    slice_x = t * np.cos(angle_rad) + n // 2
    slice_y = t * np.sin(angle_rad) + n // 2
    # Sample real and imaginary parts separately with bilinear interpolation
    real_part = map_coordinates(fourier_2d.real, [slice_y, slice_x], order=1, mode='nearest')
    imag_part = map_coordinates(fourier_2d.imag, [slice_y, slice_x], order=1, mode='nearest')
    return real_part + 1j * imag_part


# ---------------------------------------------------------------------------
# Pre-computed objects (computed once at setup to keep widgets responsive)
# ---------------------------------------------------------------------------

phantom_image      = make_phantom_image(150)
phantom_image_size = phantom_image.shape[0]
# 2-D Fourier transform of the phantom — used by the Fourier slice theorem widget
phantom_fourier_2d = np.fft.fftshift(np.fft.fft2(phantom_image))


# ===========================================================================
# Widget factories — each function creates and displays one interactive panel
# ===========================================================================

def show_test_image():
    """Display the 2-D phantom and its Fourier power spectrum."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    axes[0].imshow(phantom_image, cmap='gray', origin='lower')
    axes[0].set_title('2D test image (150×150)')
    axes[0].set_xlabel('x [px]')
    axes[0].set_ylabel('z [px]')
    axes[1].imshow(power_spectrum_2d(phantom_image), cmap='inferno', origin='lower')
    axes[1].set_title('Power spectrum')
    axes[1].axis('off')
    plt.tight_layout()
    display(figure_to_image(fig))


def tilt_explorer():
    """Show the specimen at a chosen tilt angle together with the resulting 1-D projection profile."""
    tilt_angle_slider = IntSlider(value=0, min=-70, max=70, step=1,
                                  description="Tilt angle (°)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        tilt_angle   = tilt_angle_slider.value
        rotated      = nd_rotate(phantom_image, tilt_angle, reshape=False, cval=0)
        # A 1-D projection is the column sum of the rotated image, normalised by image height
        projection   = rotated.sum(axis=0) / phantom_image.shape[0]

        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        axes[0].imshow(phantom_image, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title("Original specimen")
        axes[1].imshow(rotated, cmap='gray', origin='lower')
        axes[1].axis('off')
        axes[1].set_title(f"Specimen at α={tilt_angle}°")
        # Display the projection horizontally (intensity along x-axis, pixel index along y-axis)
        axes[2].plot(projection, np.arange(len(projection)), color='steelblue', lw=1.4)
        axes[2].set_xlim(left=0)
        axes[2].set_xlabel("Projection intensity")
        axes[2].set_ylabel("x [pixel]")
        axes[2].set_title("1D projection")
        axes[2].grid(alpha=0.3)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    tilt_angle_slider.observe(update_plot, names='value')
    display(VBox([tilt_angle_slider, output_area]))
    update_plot()


def tilt_animation_gif(angle_step=5, fps=12, hold_frames=8, dpi=70, save_path=None):
    """Build and display a GIF of the specimen sweeping through its tilt range:
    a short pause at 0°, then 0° → +70° → 0° → −70° → 0°, with a pause at the
    end. The 1-D projection profile is redrawn underneath the specimen at each
    angle. Unlike `tilt_explorer`, this plays automatically and needs no
    slider — useful for a quick, hands-off overview of the acquisition geometry.

    Raise `dpi` (e.g. 300 or 600) for a sharper render. Note that GIF is
    limited to a 256-colour palette regardless of dpi — this only improves
    pixel sharpness, not colour depth — and file size/build time grow
    roughly with dpi², so 600 dpi can take a while and produce a multi-MB
    file. Pass `save_path` to also write the GIF to disk (e.g. "tilt.gif").
    """
    step        = angle_step
    zero_to_pos = np.arange(0, 70 + step, step)      # 0, 5, ..., 70
    pos_to_zero = zero_to_pos[-2::-1]                 # 65, ..., 0
    zero_to_neg = -zero_to_pos                        # 0, -5, ..., -70 (mirror of zero_to_pos)
    neg_sweep   = zero_to_neg[1:]                      # -5, ..., -70
    neg_to_zero = zero_to_neg[-2::-1]                   # -65, ..., 0

    angles = np.concatenate([
        np.zeros(hold_frames, dtype=int),   # pause at the start
        zero_to_pos[1:], pos_to_zero,        # 0 -> +70 -> 0
        neg_sweep, neg_to_zero,               # 0 -> -70 -> 0
        np.zeros(hold_frames, dtype=int),    # pause at the end
    ])

    # Fix the projection y-axis across all frames so the plot doesn't rescale as it plays
    proj_max = max(
        (nd_rotate(phantom_image, a, reshape=False, cval=0).sum(axis=0) / phantom_image.shape[0]).max()
        for a in zero_to_pos
    ) * 1.05

    frames = []
    for angle in angles:
        rotated    = nd_rotate(phantom_image, angle, reshape=False, cval=0)
        projection = rotated.sum(axis=0) / phantom_image.shape[0]

        fig, (ax_img, ax_proj) = plt.subplots(
            2, 1, figsize=(3.8, 5.2), gridspec_kw={'height_ratios': [3, 1.3]}
        )
        ax_img.imshow(rotated, cmap='gray', origin='lower', aspect='auto',
                      vmin=phantom_image.min(), vmax=phantom_image.max())
        ax_img.axis('off')
        ax_img.set_title(f"Specimen at α = {angle:+.0f}°")

        # Projection plotted with x-axis aligned to the image's column index above it
        ax_proj.plot(projection, color='steelblue', lw=1.4)
        ax_proj.set_xlim(0, len(projection) - 1)
        ax_proj.set_ylim(0, proj_max)
        ax_proj.set_xlabel("x [pixel]")
        ax_proj.set_ylabel("Projection")
        ax_proj.grid(alpha=0.3)
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=dpi)
        plt.close(fig)
        buf.seek(0)
        frames.append(PILImage.open(buf).convert('RGB').convert('P', palette=PILImage.ADAPTIVE))

    gif_buf = io.BytesIO()
    frames[0].save(gif_buf, format='GIF', save_all=True, append_images=frames[1:],
                   duration=int(1000 / fps), loop=0)
    gif_data = gif_buf.getvalue()

    if save_path is not None:
        with open(save_path, 'wb') as f:
            f.write(gif_data)
        print(f"Saved {len(gif_data) / 1e6:.1f} MB to {save_path}")

    display(Image(data=gif_data, format='gif'))


def show_sinogram():
    """Display the full sinogram for a ±60° tilt series and a single-pixel sinusoidal trace."""
    tilt_angles_full = np.arange(-60, 61, 3)
    sinogram_full    = compute_radon_projections(phantom_image, tilt_angles_full)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(sinogram_full, cmap='gray', origin='lower', aspect='auto',
                   extent=[0, phantom_image.shape[1], tilt_angles_full[0], tilt_angles_full[-1]])
    axes[0].set_xlabel('x [px]')
    axes[0].set_ylabel('Tilt angle (°)')
    axes[0].set_title(f'Sinogram — {len(tilt_angles_full)} tilts, ±60°')

    # Trace one pixel across all tilt angles — it traces a sinusoid, hence "sinogram"
    trace_row, trace_col = 100, 50
    pixel_trace = [nd_rotate(phantom_image, a, reshape=False)[trace_row, trace_col]
                   for a in tilt_angles_full]
    axes[1].plot(tilt_angles_full, pixel_trace, color='steelblue', lw=1.4)
    axes[1].set_xlabel('Tilt angle (°)')
    axes[1].set_ylabel('Projection value')
    axes[1].set_title(f'Pixel ({trace_col},{trace_row}) vs tilt')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    display(figure_to_image(fig))


def backprojection():
    """Demonstrate simple backprojection at different tilt ranges and angular sampling."""
    max_tilt_slider      = IntSlider(value=60, min=10, max=70, step=5,
                                     description="Max tilt (°)",      style=label_style, layout=slider_layout)
    tilt_increment_slider = IntSlider(value=3,  min=1,  max=15, step=1,
                                      description="Tilt increment (°)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        tilt_angles  = np.arange(-max_tilt_slider.value, max_tilt_slider.value + 1, tilt_increment_slider.value)
        sinogram     = compute_radon_projections(phantom_image, tilt_angles)
        reconstruction = simple_backprojection(sinogram, tilt_angles)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(phantom_image, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title("True image")
        axes[1].imshow(sinogram, cmap='gray', origin='lower', aspect='auto')
        axes[1].set_title(f"Sinogram ({len(tilt_angles)} tilts)")
        axes[1].set_xlabel("x")
        axes[1].set_ylabel("Tilt index")
        axes[2].imshow(reconstruction, cmap='gray', origin='lower')
        axes[2].axis('off')
        axes[2].set_title(f"Backprojection\n(±{max_tilt_slider.value}°, Δ={tilt_increment_slider.value}°)")
        axes[3].imshow(power_spectrum_2d(reconstruction), cmap='inferno', origin='lower')
        axes[3].axis('off')
        axes[3].set_title("Fourier space of reconstruction")
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [max_tilt_slider, tilt_increment_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([max_tilt_slider, tilt_increment_slider, output_area]))
    update_plot()


def filtered_bp():
    """Compare simple backprojection with filtered backprojection (FBP) under a
    choice of FBP filters, and their Fourier spectra."""
    max_tilt_slider      = IntSlider(value=60, min=10, max=70, step=5,
                                     description="Max tilt (°)",      style=label_style, layout=slider_layout)
    tilt_increment_slider = IntSlider(value=3,  min=1,  max=15, step=1,
                                      description="Tilt increment (°)", style=label_style, layout=slider_layout)
    filter_dropdown = Dropdown(
        options=[(label, key) for key, label in FBP_FILTERS.items()], value='ramp',
        description="Filter", style=label_style, layout=slider_layout,
    )
    output_area = Output()

    def update_plot(_=None):
        tilt_angles       = np.arange(-max_tilt_slider.value, max_tilt_slider.value + 1, tilt_increment_slider.value)
        sinogram          = compute_radon_projections(phantom_image, tilt_angles)
        bp_reconstruction = simple_backprojection(sinogram, tilt_angles)
        filter_name       = filter_dropdown.value
        filtered_sinogram, filter_response, filter_freqs = apply_fbp_filter(sinogram, filter_name)
        fbp_reconstruction = simple_backprojection(filtered_sinogram, tilt_angles)

        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        axes[0, 0].imshow(phantom_image,     cmap='gray', origin='lower')
        axes[0, 0].axis('off')
        axes[0, 0].set_title("True image")
        axes[0, 1].imshow(bp_reconstruction,  cmap='gray', origin='lower')
        axes[0, 1].axis('off')
        axes[0, 1].set_title(f"Backprojection\n(±{max_tilt_slider.value}°)")
        axes[0, 2].imshow(fbp_reconstruction, cmap='gray', origin='lower')
        axes[0, 2].axis('off')
        axes[0, 2].set_title(f"Filtered BP\n({FBP_FILTERS[filter_name]})")

        # Bottom-left: the chosen filter's frequency response, ramp shown for reference
        axes[1, 0].plot(filter_freqs, filter_freqs, color='gray', lw=1.2, ls='--', label='Ram-Lak (ramp)')
        if filter_name != 'ramp':
            axes[1, 0].plot(filter_freqs, filter_response, color='steelblue', lw=1.8, label=FBP_FILTERS[filter_name])
        axes[1, 0].set_xlabel("Frequency ν")
        axes[1, 0].set_ylabel("|H(ν)|")
        axes[1, 0].set_title("Filter frequency response")
        axes[1, 0].legend(fontsize=8)
        axes[1, 0].grid(alpha=0.3)

        axes[1, 1].imshow(power_spectrum_2d(bp_reconstruction),  cmap='inferno', origin='lower')
        axes[1, 1].axis('off')
        axes[1, 1].set_title("BP Fourier space")
        axes[1, 2].imshow(power_spectrum_2d(fbp_reconstruction), cmap='inferno', origin='lower')
        axes[1, 2].axis('off')
        axes[1, 2].set_title("FBP Fourier space")
        plt.suptitle("BP vs filtered BP", fontsize=10)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [max_tilt_slider, tilt_increment_slider, filter_dropdown]:
        widget.observe(update_plot, names='value')
    display(VBox([max_tilt_slider, tilt_increment_slider, filter_dropdown, output_area]))
    update_plot()


def fourier_slice():
    """Verify the Fourier slice theorem: FT of projection = central slice of 2-D FT."""
    fourier_slice_tilt_slider = IntSlider(value=0, min=-70, max=70, step=1,
                                          description="Tilt angle (°)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        tilt_angle = fourier_slice_tilt_slider.value
        rotated    = nd_rotate(phantom_image, tilt_angle, reshape=False, cval=0)
        # Compute the 1-D projection and its Fourier transform
        projection_1d      = rotated.sum(axis=0) / phantom_image_size
        fourier_projection = np.fft.fftshift(np.fft.fft(projection_1d))
        projection_ft_amp  = np.log(np.abs(fourier_projection) + 1)

        # Extract the corresponding central slice from the 2-D Fourier transform
        central_slice     = extract_central_fourier_slice(phantom_fourier_2d, tilt_angle)
        central_slice_amp = np.log(np.abs(central_slice) + 1)

        # Coordinates of the central slice line in the 2-D Fourier plot
        n         = phantom_image_size
        t         = np.linspace(-n // 2, n // 2, n)
        angle_rad = np.deg2rad(tilt_angle)
        slice_x   = t * np.cos(angle_rad) + n // 2
        slice_y   = t * np.sin(angle_rad) + n // 2

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(rotated, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title(f"Specimen at α={tilt_angle}°")
        # Plot projection horizontally
        axes[1].plot(projection_1d, np.arange(len(projection_1d)), color='steelblue', lw=1.4)
        axes[1].set_xlabel("Projection")
        axes[1].set_ylabel("x")
        axes[1].set_title("1D projection")
        axes[1].grid(alpha=0.3)
        # Overlay the central slice line on the 2-D power spectrum
        axes[2].imshow(np.log(np.abs(phantom_fourier_2d) + 1), cmap='inferno', origin='lower')
        axes[2].plot(slice_x, slice_y, 'w-', lw=1.5, alpha=0.8)
        axes[2].set_title(f"2D FT + slice at {tilt_angle}°")
        axes[2].axis('off')
        # Compare 1-D FT of projection (blue) with extracted central slice (red dashed)
        axes[3].plot(projection_ft_amp, label='FT of projection',         color='steelblue', lw=1.4)
        axes[3].plot(central_slice_amp, label='Central slice of 2-D FT',  color='red',       lw=1.4, ls='--')
        axes[3].set_title("Fourier slice theorem")
        axes[3].set_xlabel("Frequency index")
        axes[3].legend(fontsize=8)
        axes[3].grid(alpha=0.3)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    fourier_slice_tilt_slider.observe(update_plot, names='value')
    display(VBox([fourier_slice_tilt_slider, output_area]))
    update_plot()


def missing_wedge():
    """Show the Fourier coverage gap (missing wedge) that results from a limited tilt range."""
    max_tilt_slider      = IntSlider(value=60, min=10, max=90, step=5,
                                     description="Max tilt (°)",      style=label_style, layout=slider_layout)
    tilt_increment_slider = IntSlider(value=3,  min=1,  max=10, step=1,
                                      description="Tilt increment (°)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        n           = phantom_image.shape[0]
        tilt_angles = np.arange(-max_tilt_slider.value, max_tilt_slider.value + 1, tilt_increment_slider.value)
        sinogram    = compute_radon_projections(phantom_image, tilt_angles)
        fbp_reconstruction = filtered_backprojection(sinogram, tilt_angles)

        # Mark the Fourier pixels that are covered by at least one central slice
        fourier_coverage = np.zeros((n, n))
        pixel_indices    = np.linspace(-n // 2, n // 2, n).astype(int)
        for tilt_angle in tilt_angles:
            angle_rad = np.deg2rad(tilt_angle)
            covered_x = np.clip((pixel_indices * np.cos(angle_rad) + n // 2).astype(int), 0, n - 1)
            covered_y = np.clip((pixel_indices * np.sin(angle_rad) + n // 2).astype(int), 0, n - 1)
            fourier_coverage[covered_y, covered_x] = 1

        # Draw lines at the edges of the missing wedge on the power spectrum
        centre = n // 2
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(phantom_image, cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title("True image")
        axes[1].imshow(fbp_reconstruction, cmap='gray', origin='lower')
        axes[1].axis('off')
        axes[1].set_title(f"FBP (±{max_tilt_slider.value}°, Δ={tilt_increment_slider.value}°)")
        axes[2].imshow(gaussian_filter(fourier_coverage, 1.5), cmap='Blues', origin='lower')
        axes[2].axis('off')
        axes[2].set_title(f"Fourier coverage ({len(tilt_angles)} tilts)")
        # Overlay missing-wedge boundaries on the FBP power spectrum
        for sign in [+1, -1]:
            wedge_angle = np.deg2rad(sign * max_tilt_slider.value + 90)
            t_line      = np.linspace(-n // 2, n // 2, n)
            axes[3].plot(t_line * np.cos(wedge_angle) + centre,
                         t_line * np.sin(wedge_angle) + centre,
                         'r--', lw=1.2)
        axes[3].imshow(power_spectrum_2d(fbp_reconstruction), cmap='inferno', origin='lower')
        axes[3].set_title("FBP Fourier space\n(red = missing wedge)")
        axes[3].axis('off')
        plt.suptitle(f"Missing wedge: tilt range ±{max_tilt_slider.value}°", fontsize=10)
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [max_tilt_slider, tilt_increment_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([max_tilt_slider, tilt_increment_slider, output_area]))
    update_plot()


def crowther_criterion():
    """Explore the Crowther criterion: how many tilt angles are needed for a given resolution?

    The criterion m ≥ πD/r says that to achieve resolution r from an object of
    diameter D, we need at least πD/r projections (for full angular coverage).
    """
    diameter_slider   = IntSlider(  value=10,  min=2,   max=50,  step=1,
                                    description="Diameter D (nm)",    style=label_style, layout=slider_layout)
    resolution_slider = FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5,
                                    description="Resolution r (nm)",  style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        diameter   = diameter_slider.value
        resolution = resolution_slider.value
        # Minimum number of tilt angles from the Crowther criterion
        min_num_tilts = int(np.ceil(np.pi * diameter / resolution))

        diameter_range   = np.linspace(2, 50, 200)
        reference_resolutions = [0.5, 1.0, 2.0, 3.0, 5.0]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        # Left: Crowther criterion curves for different target resolutions
        for ref_resolution in reference_resolutions:
            axes[0].plot(diameter_range, np.pi * diameter_range / ref_resolution,
                         lw=1.6, label=f"r={ref_resolution} nm")
        axes[0].axvline(diameter,      color='k',   ls='--', lw=1.2)
        axes[0].axhline(min_num_tilts, color='red', ls=':',  lw=1.2, label=f"m={min_num_tilts}")
        axes[0].set_xlabel("Diameter D (nm)")
        axes[0].set_ylabel("Min. tilts m")
        axes[0].set_title("Crowther criterion: m ≥ πD/r")
        axes[0].legend(fontsize=8)
        axes[0].grid(alpha=0.3)

        # Right: does the tilt series actually reach the target resolution shell?
        # Each central section (one per tilt) is a full line through the origin
        # in Fourier space; at the target shell |k| = 1/r, adjacent samples must
        # be no farther apart than the object's Nyquist spacing 1/D or a gap
        # opens up. Grey dots mark that required spacing; red dots mark where
        # the m tilts (blue lines) actually cross the shell — a smaller D means
        # sparser grey dots, so fewer tilts are needed to keep up with them.
        k_target   = 1 / resolution
        axis_limit = k_target * 1.4
        tilt_angles_full = np.linspace(-90, 90, min_num_tilts, endpoint=False)

        # Each central section crosses the shell at two antipodal points
        crossing_angles_deg = np.concatenate([tilt_angles_full, tilt_angles_full + 180])
        crossing_x = k_target * np.cos(np.deg2rad(crossing_angles_deg))
        crossing_y = k_target * np.sin(np.deg2rad(crossing_angles_deg))

        # Reference points spaced exactly 1/D apart around the full shell
        n_required = max(int(round(2 * np.pi * k_target * diameter)), 1)
        required_angles_deg = np.linspace(0, 360, n_required, endpoint=False)
        required_x = k_target * np.cos(np.deg2rad(required_angles_deg))
        required_y = k_target * np.sin(np.deg2rad(required_angles_deg))

        # Draw all central-section lines in one call (NaN-separated segments)
        line_x, line_y = [], []
        for angle in tilt_angles_full:
            angle_rad = np.deg2rad(angle)
            line_x += [-axis_limit * np.cos(angle_rad), axis_limit * np.cos(angle_rad), np.nan]
            line_y += [-axis_limit * np.sin(angle_rad), axis_limit * np.sin(angle_rad), np.nan]

        ax = axes[1]
        ax.plot(line_x, line_y, color='steelblue', lw=0.6, alpha=0.5)
        theta = np.linspace(0, 2 * np.pi, 200)
        ax.plot(k_target * np.cos(theta), k_target * np.sin(theta), 'k--', lw=1.2)
        ax.scatter(required_x, required_y, s=22, color='lightgray', edgecolor='gray', lw=0.4,
                  zorder=3, label=f"needed every 1/D = {1/diameter:.2f} nm⁻¹")
        ax.scatter(crossing_x, crossing_y, s=14, color='red', zorder=4,
                  label=f"{min_num_tilts} tilts")
        ax.set_xlim(-axis_limit, axis_limit)
        ax.set_ylim(-axis_limit, axis_limit)
        ax.set_aspect('equal')
        ax.set_xlabel("$k_x$ (1/nm)")
        ax.set_ylabel("$k_y$ (1/nm)")
        ax.set_title(f"Sampling at the resolution shell |k| = 1/r = {k_target:.2f} nm⁻¹")
        ax.legend(fontsize=7, loc='upper right')
        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    for widget in [diameter_slider, resolution_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([diameter_slider, resolution_slider, output_area]))
    update_plot()


def radon_explorer():
    """Build the sinogram and backprojection one angle at a time.

    Precomputes all rotations at setup so dragging the slider is instant.
    Panels (left to right):
      1. Tilted specimen at the current angle
      2. Sinogram built so far — current row highlighted in red
      3. Back-projected slab: the 1D projection spread back into 2D (the
         elementary inverse step that sums to the full reconstruction)
      4. Cumulative reconstruction (sum and average of all slabs so far)
    """
    tilt_angles = np.arange(-60, 61, 3)   # 41 angles, ±60°
    N = len(tilt_angles)
    n = phantom_image.shape[0]

    # Pre-compute everything once so the slider update is O(1)
    rotated_imgs, projs, slabs_back, cum_bp = [], [], [], []
    running = np.zeros((n, n))
    for i, angle in enumerate(tilt_angles):
        rot  = nd_rotate(phantom_image, angle, reshape=False, cval=0)
        proj = rot.sum(axis=0)
        slab = np.tile(proj[np.newaxis, :], (n, 1)) / n
        back = nd_rotate(slab, -angle, reshape=False, cval=0)
        running = running + back
        rotated_imgs.append(rot)
        projs.append(proj)
        slabs_back.append(back)
        cum_bp.append(running.copy() / (i + 1))

    sinogram = np.vstack([p[np.newaxis, :] for p in projs])
    sin_vmax = sinogram.max()
    slab_vmax = max(slabs_back[0].max(), 1e-10)
    recon_vmax = cum_bp[-1].max() + 1e-10

    angle_slider = IntSlider(value=1, min=1, max=N, step=1,
                             description="Projections", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        k     = angle_slider.value
        alpha = tilt_angles[k - 1]

        sin_partial = np.zeros_like(sinogram)
        sin_partial[:k] = sinogram[:k]

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

        axes[0].imshow(rotated_imgs[k - 1], cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title(f"Specimen  (α = {alpha:+d}°)")

        axes[1].imshow(sin_partial, cmap='gray', origin='lower', aspect='auto',
                       extent=[0, n, tilt_angles[0], tilt_angles[-1]],
                       vmin=0, vmax=sin_vmax)
        axes[1].axhline(alpha, color='tomato', lw=2.5, alpha=0.9)
        axes[1].set_xlabel("x [px]")
        axes[1].set_ylabel("Tilt angle (°)")
        axes[1].set_title(f"Sinogram  ({k} / {N} angles)")

        axes[2].imshow(slabs_back[k - 1], cmap='gray', origin='lower',
                       vmin=0, vmax=slab_vmax)
        axes[2].axis('off')
        axes[2].set_title(f"Back-projection slab  (α = {alpha:+d}°)")

        axes[3].imshow(cum_bp[k - 1], cmap='gray', origin='lower',
                       vmin=0, vmax=recon_vmax)
        axes[3].axis('off')
        axes[3].set_title(f"Reconstruction  ({k} angles)")

        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    angle_slider.observe(update_plot, names='value')
    display(VBox([angle_slider, output_area]))
    update_plot()


def radon_explorer_gif(hold_ms=500, final_hold_ms=1500, dpi=70, save_path=None):
    """Build and display a GIF that steps through the same build-up as
    `radon_explorer`: at each of the pre-set tilt angles it shows the tilted
    specimen, the sinogram accumulated so far (current row highlighted in
    red), the back-projection slab, and the cumulative reconstruction. Each
    tilt is held on screen for `hold_ms` milliseconds so the sinogram and
    reconstruction visibly build up step by step instead of flashing past;
    the final, complete reconstruction is held for `final_hold_ms` before the
    GIF loops back to the start. Pass `save_path` to also write the GIF to
    disk. See `tilt_animation_gif` for notes on raising `dpi` for a sharper
    render.
    """
    tilt_angles = np.arange(-60, 61, 3)   # 41 angles, ±60° — same series as radon_explorer
    N = len(tilt_angles)
    n = phantom_image.shape[0]

    # Pre-compute every step once, exactly as in radon_explorer
    rotated_imgs, projs, slabs_back, cum_bp = [], [], [], []
    running = np.zeros((n, n))
    for i, angle in enumerate(tilt_angles):
        rot  = nd_rotate(phantom_image, angle, reshape=False, cval=0)
        proj = rot.sum(axis=0)
        slab = np.tile(proj[np.newaxis, :], (n, 1)) / n
        back = nd_rotate(slab, -angle, reshape=False, cval=0)
        running = running + back
        rotated_imgs.append(rot)
        projs.append(proj)
        slabs_back.append(back)
        cum_bp.append(running.copy() / (i + 1))

    sinogram   = np.vstack([p[np.newaxis, :] for p in projs])
    sin_vmax   = sinogram.max()
    slab_vmax  = max(slabs_back[0].max(), 1e-10)
    recon_vmax = cum_bp[-1].max() + 1e-10

    frames = []
    for k in range(1, N + 1):
        alpha = tilt_angles[k - 1]

        sin_partial = np.zeros_like(sinogram)
        sin_partial[:k] = sinogram[:k]

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

        axes[0].imshow(rotated_imgs[k - 1], cmap='gray', origin='lower')
        axes[0].axis('off')
        axes[0].set_title(f"Specimen  (α = {alpha:+d}°)")

        axes[1].imshow(sin_partial, cmap='gray', origin='lower', aspect='auto',
                       extent=[0, n, tilt_angles[0], tilt_angles[-1]],
                       vmin=0, vmax=sin_vmax)
        axes[1].axhline(alpha, color='tomato', lw=2.5, alpha=0.9)
        axes[1].set_xlabel("x [px]")
        axes[1].set_ylabel("Tilt angle (°)")
        axes[1].set_title(f"Sinogram  ({k} / {N} angles)")

        axes[2].imshow(slabs_back[k - 1], cmap='gray', origin='lower',
                       vmin=0, vmax=slab_vmax)
        axes[2].axis('off')
        axes[2].set_title(f"Back-projection slab  (α = {alpha:+d}°)")

        axes[3].imshow(cum_bp[k - 1], cmap='gray', origin='lower',
                       vmin=0, vmax=recon_vmax)
        axes[3].axis('off')
        axes[3].set_title(f"Reconstruction  ({k} angles)")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=dpi)
        plt.close(fig)
        buf.seek(0)
        frames.append(PILImage.open(buf).convert('RGB').convert('P', palette=PILImage.ADAPTIVE))

    # Per-frame duration list: every step held for hold_ms, the final one longer
    durations = [hold_ms] * (N - 1) + [final_hold_ms]
    gif_buf = io.BytesIO()
    frames[0].save(gif_buf, format='GIF', save_all=True, append_images=frames[1:],
                   duration=durations, loop=0)
    gif_data = gif_buf.getvalue()

    if save_path is not None:
        with open(save_path, 'wb') as f:
            f.write(gif_data)
        print(f"Saved {len(gif_data) / 1e6:.1f} MB to {save_path}")

    display(Image(data=gif_data, format='gif'))



def tilt_thickness_widget():
    """Show how specimen tilt increases the projected path length (1/cos law),
    introduces a defocus gradient across the image, and damps high-resolution
    Thon rings more than low-resolution ones."""
    from matplotlib.patches import Polygon as _MplPoly

    # Physical parameters
    B_base     = 50.0     # B-factor at zero tilt (Ang^2) — thickness contribution
    defocus_A  = 20000.0  # Central defocus (Ang)
    lam        = 0.0197   # Electron wavelength at 300 kV (Ang)
    n_img      = 4096     # Realistic image width (px) for defocus gradient panel
    pixel_size = 2.0      # Ang/pixel
    nim_2d     = 128      # Size of 2D Thon ring display

    half_width_A = (n_img / 2) * pixel_size

    # Pre-compute 2D k grid and CTF^2 (independent of tilt)
    kc      = np.fft.fftshift(np.fft.fftfreq(nim_2d, d=pixel_size))
    k_2d    = np.sqrt(kc[:, None]**2 + kc[None, :]**2)
    ctf2_2d = np.sin(np.pi * lam * defocus_A * k_2d**2) ** 2

    # Pre-compute the fixed alpha=0 reference image
    E0_2d    = np.exp(-B_base * k_2d**2)
    thon_ref = ctf2_2d * E0_2d**2

    # Half-power cutoff at zero tilt (for reference circle)
    k_cut_0  = np.sqrt(np.log(2) / (2 * B_base))
    r_cut_0  = k_cut_0 * nim_2d * pixel_size

    # CTF map axes for the defocus gradient panel
    n_x_map = 220
    n_k_map = 140
    x_map_A = np.linspace(-half_width_A, half_width_A, n_x_map)
    k_map   = np.linspace(0, 0.25, n_k_map)
    theta_c = np.linspace(0, 2 * np.pi, 300)
    cx = cy = nim_2d / 2

    tilt_slider = IntSlider(value=0, min=0, max=70, step=5,
                            description="Tilt (deg)", style=label_style, layout=slider_layout)
    output_area = Output()

    def update_plot(_=None):
        alpha_deg = tilt_slider.value
        alpha_rad = np.deg2rad(max(alpha_deg, 0.01))
        cos_a = np.cos(alpha_rad)
        sin_a = np.sin(alpha_rad)

        B_eff = B_base / cos_a

        # Thon rings at current tilt
        E_2d     = np.exp(-B_eff * k_2d**2)
        thon_cur = ctf2_2d * E_2d**2

        # Half-power cutoff at current tilt
        k_cut = np.sqrt(np.log(2) / (2 * B_eff))
        r_cut = k_cut * nim_2d * pixel_size

        # CTF map: vectorised over x positions
        phase_map = (np.pi * lam
                     * (defocus_A + x_map_A[np.newaxis, :] * sin_a)
                     * k_map[:, np.newaxis]**2)
        ctf_map = np.sin(phase_map)**2

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

        # Panel 1 — slab cross-section with beam path ──────────────────────
        ax = axes[0]
        ax.set_xlim(-1.1, 1.1)
        ax.set_ylim(-0.85, 0.85)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f"Sample cross-section  (α = {alpha_deg}°)", pad=8)

        t, W = 0.25, 0.88
        corners = np.array([[-W/2, -t/2], [W/2, -t/2], [W/2, t/2], [-W/2, t/2]])
        R   = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        rot = (R @ corners.T).T

        slab = _MplPoly(rot, closed=True, fc='steelblue', alpha=0.45, ec='#6aaddd', lw=2)
        ax.add_patch(slab)

        ys = []
        for i in range(4):
            p1, p2 = rot[i], rot[(i + 1) % 4]
            dx = p2[0] - p1[0]
            if abs(dx) > 1e-9:
                u = -p1[0] / dx
                if 0.0 <= u <= 1.0:
                    ys.append(p1[1] + u * (p2[1] - p1[1]))
        if len(ys) >= 2:
            ys.sort()
            y_lo, y_hi = ys[0], ys[-1]
            ax.plot([0, 0], [y_lo, y_hi], '-', color='orange', lw=4, solid_capstyle='round')
            ax.annotate('', xy=(0.18, y_lo), xytext=(0.18, y_hi),
                        arrowprops=dict(arrowstyle='<->', color='orange', lw=1.5))
            label = (f't₀/cosα = {1/cos_a:.2f}×t₀'
                     if alpha_deg > 0 else 't₀')
            ax.text(0.25, (y_lo + y_hi) / 2, label, va='center', fontsize=9, color='orange')

        ax.annotate('', xy=(-0.60, -t/2), xytext=(-0.60, t/2),
                    arrowprops=dict(arrowstyle='<->', color='#aaaaaa', lw=1.2))
        ax.text(-0.82, 0, 't₀', va='center', ha='center', fontsize=9, color='#aaaaaa')

        if alpha_deg > 0:
            theta_arc = np.linspace(0, alpha_rad, 60)
            ax.plot(0.48 * np.sin(theta_arc), -0.48 * np.cos(theta_arc),
                    color='#ffff80', lw=1.5, alpha=0.8)
            mid = alpha_rad / 2
            ax.text(0.26 * np.sin(mid), -0.26 * np.cos(mid), 'α',
                    ha='center', va='center', fontsize=11, color='#ffff80', style='italic')

        for xb in np.linspace(-0.7, 0.7, 8):
            ax.annotate('', xy=(xb, -0.82), xytext=(xb, 0.82),
                        arrowprops=dict(arrowstyle='->', color='#90ee90', lw=1.1, alpha=0.35))

        # Panel 2 — CTF map across the tilted image (defocus gradient) ──────
        ax = axes[1]
        x_nm = x_map_A / 10
        ax.imshow(ctf_map, cmap='gray', origin='lower', aspect='auto',
                  extent=[x_nm[0], x_nm[-1], 0, 0.25],
                  vmin=0, vmax=1)
        ax.set_xlabel("Image position (nm)")
        ax.set_ylabel("Spatial frequency (Å⁻¹)")
        df_range_nm = half_width_A * sin_a / 10
        ax.set_title(f"CTF across tilted image\nΔf₀ ± {df_range_nm:.0f} nm  (α = {alpha_deg}°)")

        # Panel 3 — reference Thon rings at α = 0° (fixed) ──────────────────
        ax = axes[2]
        ax.imshow(thon_ref, cmap='inferno', origin='lower', vmin=0, vmax=1)
        ax.axis('off')
        ax.plot(cx + r_cut_0 * np.cos(theta_c),
                cy + r_cut_0 * np.sin(theta_c),
                '--', color='white', lw=1.5, alpha=0.9)
        ax.set_title(f"Thon rings — α = 0°\n(cut-off {1/k_cut_0:.0f} Å, dashed)")

        # Panel 4 — Thon rings at current tilt ───────────────────────────────
        ax = axes[3]
        ax.imshow(thon_cur, cmap='inferno', origin='lower', vmin=0, vmax=1)
        ax.axis('off')
        ax.plot(cx + r_cut * np.cos(theta_c),
                cy + r_cut * np.sin(theta_c),
                '--', color='white', lw=1.5, alpha=0.9)
        ax.set_title(f"Thon rings — α = {alpha_deg}°\n(cut-off {1/k_cut:.0f} Å, dashed)")

        plt.tight_layout()
        with output_area:
            clear_output(wait=True)
            display(figure_to_image(fig))

    tilt_slider.observe(update_plot, names='value')
    display(VBox([tilt_slider, output_area]))
    update_plot()

print("Setup complete.")


# Practical: Electron Tomography

This practical covers tomographic reconstruction: backprojection, filtered backprojection, the Fourier slice theorem, and the Crowther criterion.

> **To start the practicals:** click **Run All** (⏭) in the toolbar above.

***

## The 2D test image

We work with a 2D phantom that mimics a biological cross-section: a hollow **ring** (microtubule), two **line segments** (membrane sheets), and a small **dot** (protein complex).

In [ ]:
show_test_image()

***

## Tilting the sample and acquiring projections

To acquire a tilt series, the specimen is physically rotated about the $y$-axis. At each tilt angle $\alpha$, the transmitted electrons form a 1D projection (the column sum of the rotated image). The interactive below shows the specimen at a chosen tilt angle together with the resulting 1D projection profile.

In [ ]:
tilt_explorer()

For a hands-off overview of the same relationship, here is the specimen rotating continuously through its full ±70° tilt range, with the 1D projection redrawn underneath it as it goes:

In [ ]:
tilt_animation_gif()

***

## Signal attenuation with tilt: the 1/cosα law

When the specimen is tilted by angle α, the electron beam must traverse a longer
path through the specimen:

$$t_\text{eff}(\alpha) = \frac{t_0}{\cos\alpha}$$

where $t_0$ is the physical specimen thickness measured perpendicular to the
specimen plane. The fraction of coherently scattered electrons falls exponentially
with path length ($\propto e^{-t_\text{eff}/\lambda_\text{MFP}}$, where
$\lambda_\text{MFP}$ is the inelastic mean free path, typically ~35 nm in vitreous
ice at 300 kV), so high-tilt images have substantially weaker coherent signal.
This is visible as a reduction in Thon ring contrast in the power spectrum of
those images.


In [ ]:
tilt_thickness_widget()

***

## The sinogram and backprojection

Collecting projections over the full tilt range and stacking them row by row produces the
**sinogram** — so named because a point object traces a sinusoidal curve across it as the
tilt angle varies.

The interactive below lets you step through the tilt series one angle at a time and watch
four things happen simultaneously:

1. **Tilted specimen** — the image the detector records at each angle.
2. **Sinogram** — each new projection adds one row; the current row is highlighted in red.
3. **Back-projection slab** — the elementary inverse step: the 1D projection is spread
   uniformly back into 2D space along the projection direction. Summing these slabs over
   all angles reconstructs the object.
4. **Cumulative reconstruction** — the running average of all slabs so far. Notice how
   the image gradually sharpens as more angles are added, and that features already visible
   with few angles become more defined as angular coverage improves.

> With only a handful of angles the reconstruction is a blurry superposition of smeared
> stripes — the star artefact of unfiltered backprojection. Adding more angles fills
> Fourier space more uniformly and sharpens the result.

In [ ]:
radon_explorer()

As a hands-off overview of the same build-up, here it is as a GIF: each tilt is held on screen for a moment so you can watch the sinogram fill in row by row and the reconstruction sharpen, without having to drag the slider yourself.

In [ ]:
radon_explorer_gif()

***

## Task 1 – Backprojection: effect of angular sampling

You have seen individual back-projection slabs accumulate into a reconstruction. The quality of that reconstruction depends critically on two parameters: the **tilt range** (determines the missing wedge) and the **angular increment** (determines how densely Fourier space is sampled). The simplest reconstruction formula is:

$$
A_\text{BP}(x,z) = \sum_\alpha p_\alpha\!\left(x\cos\alpha + z\sin\alpha\right)
$$

This is correct in principle but overweights low-spatial-frequency components (every point in the image receives contributions from every projection), producing a blurry, star-shaped reconstruction artifact.

```{admonition} Task 1
:class: note
1. Take a 1D projection `projection`.
2. Repeat it $N$ times along a new axis using `repeat_line` to get a 2D slab of shape $(N, N)$.
3. Rotate the slab back by $-\alpha$ using `nd_rotate`.
4. Sum over all angles. Divide by the number of angles.
```

In [ ]:
backprojection()

***

## Task 2 – Filtered backprojection

The blurring artifact of plain backprojection is caused by the over-sampling of low frequencies: every projection adds a constant along its direction, contributing to the low-frequency region of Fourier space more than to high frequencies. The fix is to pre-filter each projection with a **ramp filter** $|\nu|$ before backprojecting. This exactly counteracts the $1/|\nu|$ weighting introduced by backprojection.

$$
A_\text{FBP}(x,z) = \sum_\alpha \bigl[p_\alpha * h\bigr]\!\left(x\cos\alpha + z\sin\alpha\right), \qquad \hat h(\nu) = |\nu|
$$

```{admonition} Task 2
:class: note
1. Take a 1D projection.
2. Compute its Fourier transform using `np.fft.rfft`.
3. Multiply by the ramp filter $|\nu|$ (use `np.fft.rfftfreq` for the frequency axis).
4. Inverse-Fourier-transform back.
5. Use the filtered projections for backprojection.
```

In practice the pure ramp filter is rarely used unmodified: it amplifies high-frequency noise just as much as it amplifies genuine high-resolution signal, since $|\nu|$ grows without bound towards the Nyquist frequency. Real FBP implementations instead multiply the ramp by a **window** that tapers it back towards zero near Nyquist, trading a little resolution for much less noise. The interactive below lets you switch between five standard choices, from sharpest/noisiest to smoothest/most attenuated:

- **Ram-Lak** — the pure ramp, no windowing
- **Shepp-Logan** — ramp × sinc window
- **Cosine** — ramp × cosine window
- **Hamming** — ramp × raised-cosine window with a small non-zero floor at Nyquist
- **Hann** — ramp × raised-cosine window that reaches exactly zero at Nyquist

The bottom-left panel plots the selected filter's frequency response $|H(\nu)|$ against the unwindowed ramp, so you can see directly how much high-frequency content each choice keeps or discards.</cell id="a0db9754">


In [ ]:
filtered_bp()

***

## The Fourier Slice Theorem

The **projection-slice theorem** (also called the Fourier slice theorem) states that the 1D Fourier transform of a projection at angle $\alpha$ equals a central slice through the 2D Fourier transform of the object at the same angle:

$$
\mathcal{F}_1\{p_\alpha\}(\nu) = \mathcal{F}_2\{A\}(\nu\cos\alpha,\, \nu\sin\alpha)
$$

Each projection therefore fills one line (passing through the origin) in 2D Fourier space. To reconstruct the full 3D (or 2D) object, we need enough projections to cover Fourier space densely.

The interactive below shows, for a chosen tilt angle:
1. The rotated specimen and its 1D projection
2. The 2D Fourier transform of the specimen with the corresponding central slice highlighted
3. A verification that the 1D FT of the projection matches the highlighted central slice

In [ ]:
fourier_slice()

***

## Missing wedge

In practice the tilt range is limited to roughly $\pm 60°$ because:
- **Grid bars** obstruct the beam at high tilt
- **Increased effective sample thickness** at high tilt degrades image quality

This means the central region of Fourier space is never filled — the **missing wedge**. Features oriented along the tilt axis (perpendicular to the tilt direction) are most affected: they appear elongated in the reconstruction.

The interactive below shows the Fourier coverage at different tilt ranges. The missing wedge is the dark triangular gap.

In [ ]:
missing_wedge()

***

## Task 3 – The Crowther criterion

To fully reconstruct an object of diameter $D$ at resolution $r$, how many tilt angles $m$ do we need? The answer follows from the Fourier slice theorem and the sampling theorem.

Each projection fills one central line in 2D Fourier space. At resolution $r$, we need the outermost Fourier shell (radius $1/r$) to be densely sampled. Two adjacent central lines at the edge of the Fourier disk must be separated by at most $1/D$ (the Nyquist frequency across the object). The angular separation between adjacent projections at radius $1/r$ is therefore:

$$
\Delta\alpha \leq \frac{1/D}{1/r} = \frac{r}{D}
$$

Since the total range is $\pi$ radians (projections at $\alpha$ and $\alpha+\pi$ are identical), the number of tilts needed is:

$$
\boxed{m \geq \frac{\pi}{\Delta\alpha} = \frac{\pi D}{r}}
$$

This is the **Crowther criterion**. It tells us that to achieve resolution $r$ from an object of diameter $D$, we need at least $\pi D/r$ projections — independent of whether we are doing tomography or SPA (in SPA, particles in random orientations naturally fill the full angular range).

Note that $D$ scales directly with $m$: a thin or small object (small $D$) needs far fewer tilts than a large one for the same resolution, because a more compact real-space object has a broader Fourier transform, so its central sections overlap more between neighbouring angles. In a single-axis tilt series, $D$ is best thought of as the specimen thickness — the dimension the missing wedge constrains — rather than the full lateral field of view. The interactive below makes this concrete: the right-hand panel shows exactly where the $m$ tilts cross the target resolution shell (red) against the spacing $1/D$ that is actually required there (grey).</cell id="b904005b">


In [ ]:
crowther_criterion()

***

## Summary

| Concept | Key idea |
|---------|----------|
| Tilt series | Physical rotation of specimen; each tilt gives one 1D projection |
| Sinogram | Stack of 1D projections; point objects trace sinusoids |
| Radon transform | Mathematical projection operator $p_\alpha(t) = \int A(t\cos\alpha - s\sin\alpha, t\sin\alpha + s\cos\alpha)\,ds$ |
| Backprojection | Smear each projection back along its direction and sum |
| Ramp filter | $\hat h(\nu) = |\nu|$; corrects low-frequency overweighting of backprojection |
| Fourier slice theorem | $\mathcal{F}_1\{p_\alpha\}(\nu) = \mathcal{F}_2\{A\}(\nu\cos\alpha, \nu\sin\alpha)$ |
| Missing wedge | Triangular gap in Fourier coverage from limited tilt range |
| Crowther criterion | $m \geq \pi D / r$ tilts for resolution $r$ from object of diameter $D$ |